# 1e5 Msun PhenomXPy Waveform

Generate the lower-mass waveform used for the `1e5-data` LISA link simulation.


In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

try:
    from phenomxpy import IMRPhenomTHM
except ImportError as exc:
    raise ImportError("Could not import phenomxpy in this environment.") from exc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "1e5-wave":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "lisa_link_sim":
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_DIR = PROJECT_ROOT / "lisa_link_sim" / "1e5-wave"
INPUT_DIR = BASE_DIR / "inputs"
OUTPUT_DIR = BASE_DIR / "outputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WAVEFORM_FILE = INPUT_DIR / "my_waveform_1e5.npz"
METADATA_FILE = OUTPUT_DIR / "my_waveform_1e5_metadata.json"
PLOT_FILE = OUTPUT_DIR / "phenomxpy_waveform_1e5_preview.png"

# Waveform settings. The LISA simulation script will resample this onto dt=0.1 s.
raw_dt = 1.0
raw_half_duration = 300_000.0
times = np.arange(-raw_half_duration, raw_half_duration, raw_dt)

eta = 0.24
total_mass = 1.0e5
distance = 6230.0
s1 = 0.3
s2 = -0.2
f_min = 1.0e-4
iota = 0.7
phi_ref = 0.0


def build_phenom_model():
    constructor_attempts = [
        dict(eta=eta, total_mass=total_mass, distance=distance, s1=s1, s2=s2, f_min=f_min, iota=iota, phi_ref=phi_ref),
        dict(eta=eta, total_mass=total_mass, distance=distance, s1=s1, s2=s2, f_min=f_min, inclination=iota, phi_ref=phi_ref),
        dict(eta=eta, total_mass=total_mass, distance=distance, s1=s1, s2=s2, f_min=f_min),
    ]
    last_error = None
    for kwargs in constructor_attempts:
        try:
            print("Trying IMRPhenomTHM with kwargs:")
            for k, v in kwargs.items():
                print(f"  {k} = {v}")
            return IMRPhenomTHM(**kwargs), kwargs
        except (TypeError, ValueError) as exc:
            print(f"  rejected: {exc}\n")
            last_error = exc
    raise RuntimeError("Could not construct IMRPhenomTHM") from last_error


def unpack_polarizations(out, input_times):
    if not isinstance(out, (tuple, list)):
        raise TypeError(f"Expected tuple/list from compute_polarizations, got {type(out)}")
    arrays = [np.asarray(item) for item in out]
    print(f"compute_polarizations returned {len(arrays)} arrays")
    for i, arr in enumerate(arrays):
        print(f"  output[{i}]: shape={arr.shape}, min={np.nanmin(arr):.3e}, max={np.nanmax(arr):.3e}")

    if len(arrays) == 2:
        hplus, hcross = arrays
        t_save = input_times
    elif len(arrays) == 3:
        if arrays[0].shape == input_times.shape and np.all(np.diff(arrays[0]) > 0):
            t_save, hplus, hcross = arrays
        elif arrays[2].shape == input_times.shape and np.all(np.diff(arrays[2]) > 0):
            hplus, hcross, t_save = arrays
        else:
            t_save = input_times
            hplus, hcross = arrays[:2]
    else:
        raise ValueError(f"Unexpected number of outputs: {len(arrays)}")

    t_save = np.asarray(t_save, dtype=float)
    hplus = np.asarray(hplus, dtype=float)
    hcross = np.asarray(hcross, dtype=float)
    if not (len(t_save) == len(hplus) == len(hcross)):
        raise ValueError("Unpacked waveform arrays do not have matching lengths")
    return t_save, hplus, hcross


print("Generating 1e5 Msun waveform with phenomxpy IMRPhenomTHM...")
phen, used_kwargs = build_phenom_model()
out = phen.compute_polarizations(times)
t_save, hplus, hcross = unpack_polarizations(out, times)

bad = ~np.isfinite(t_save) | ~np.isfinite(hplus) | ~np.isfinite(hcross)
if np.any(bad):
    print(f"Removing {bad.sum()} non-finite samples")
    keep = ~bad
    t_save = t_save[keep]
    hplus = hplus[keep]
    hcross = hcross[keep]

amp = np.sqrt(hplus**2 + hcross**2)
peak_index = int(np.argmax(amp))
peak_time = float(t_save[peak_index])
threshold = 1e-3 * float(np.max(amp))
idx = np.flatnonzero(amp >= threshold)
signal_window = [float(t_save[idx[0]]), float(t_save[idx[-1]])] if len(idx) else [None, None]

np.savez(
    WAVEFORM_FILE,
    t=t_save,
    hplus=hplus,
    hcross=hcross,
    total_mass=np.asarray(total_mass),
    distance_mpc=np.asarray(distance),
    f_min_hz=np.asarray(f_min),
)
metadata = {
    "eta": eta,
    "total_mass": total_mass,
    "distance_mpc": distance,
    "s1": s1,
    "s2": s2,
    "f_min_hz": f_min,
    "iota": iota,
    "phi_ref": phi_ref,
    "raw_dt_s": raw_dt,
    "raw_half_duration_s": raw_half_duration,
    "n_samples": int(len(t_save)),
    "peak_time_s": peak_time,
    "max_strain": float(np.max(amp)),
    "threshold_1e-3_window_s": signal_window,
    "constructor_kwargs": used_kwargs,
}
METADATA_FILE.write_text(json.dumps(metadata, indent=2))

print("Saved waveform:", WAVEFORM_FILE)
print("Saved metadata:", METADATA_FILE)
print(f"N={len(t_save)}, dt={np.median(np.diff(t_save)):.6g} s, range=[{t_save.min():.1f}, {t_save.max():.1f}] s")
print(f"peak_time={peak_time:.3f} s, max_strain={np.max(amp):.3e}")
print(f"1e-3 amplitude window={signal_window}")

plt.figure(figsize=(12, 4))
plt.plot(t_save / 86400.0, hplus, label="hplus", lw=0.8)
plt.plot(t_save / 86400.0, hcross, label="hcross", lw=0.8, alpha=0.8)
plt.axvline(peak_time / 86400.0, color="k", ls="--", alpha=0.5, label="peak")
plt.xlabel("time [days]")
plt.ylabel("strain")
plt.title("PhenomXPy waveform, total mass 1e5 Msun, f_min=1e-4 Hz")
plt.grid(alpha=0.3)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=150)
plt.show()
print("Saved plot:", PLOT_FILE)


Generating 1e5 Msun waveform with phenomxpy IMRPhenomTHM...
Trying IMRPhenomTHM with kwargs:
  eta = 0.24
  total_mass = 100000.0
  distance = 6230.0
  s1 = 0.3
  s2 = -0.2
  f_min = 0.0001
  iota = 0.7
  phi_ref = 0.0


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


compute_polarizations returned 3 arrays
  output[0]: shape=(600000,), min=-1.736e-19, max=1.659e-19
  output[1]: shape=(600000,), min=-1.683e-19, max=1.744e-19
  output[2]: shape=(600000,), min=-3.000e+05, max=3.000e+05
Saved waveform: /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/1e5-wave/inputs/my_waveform_1e5.npz
Saved metadata: /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/1e5-wave/outputs/my_waveform_1e5_metadata.json
N=600000, dt=1 s, range=[-300000.0, 299999.0] s
peak_time=0.000 s, max_strain=1.821e-19
1e-3 amplitude window=[-300000.0, 45.0]
Saved plot: /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/1e5-wave/outputs/phenomxpy_waveform_1e5_preview.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23278/864567518.py:156: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
